### Project DRISHTI: Lunar South Pole Navigation & Hazard Pipeline
**Objective:** Map traversable terrain, identify Permanently Shadowed Regions (PSRs), and compute an optimal rover path to subsurface water-ice inside the Shoemaker Crater region using DFSAR, LOLA, and LRO illumination data.

---

#### Phase 0 & 1: Environment Initialization and Spatial Overlap Audit
**Scientific Rationale:** Before applying any gradient mathematics or morphological filters, we must mathematically guarantee that our Topography (DEM) and Illumination datasets geographically encompass our target Radar Swath (CPR/Ice Map). 

Warping a small regional tile into a large radar bounding box causes GDAL to pad the void space with `0.0` values. In planetary GIS, algorithms interpret `0.0` elevation or slope as perfectly safe, flat terrain, leading to catastrophic false-positive landing site selections (the "Ghost Pixel" problem). 

This cell initializes our strict `-9999.0` NoData environment and audits the bounding boxes of the raw PDS/PGDA files against the DFSAR radar swath to confirm 100% true spatial overlap.

In [1]:
import rasterio
import numpy as np
from scipy.ndimage import (distance_transform_edt, binary_dilation, 
                           label, maximum_filter, minimum_filter,
                           generic_filter)
import os

# --- GLOBAL CONSTANTS ---
NODATA = -9999.0
BASE_DIR = r"C:\DRISHTI_POC"


def spatial_overlap_audit_v2():
    print("=" * 70)
    print("PHASE 1: SPATIAL OVERLAP AUDIT (Raw Source vs Radar Swath)")
    print("=" * 70)
    
    # Files we are checking
    files = {
        "DFSAR_RADAR_SWATH": os.path.join(BASE_DIR, "WARPED_CPR.tif"),
        "ICE_TARGETS": os.path.join(BASE_DIR, "DRISHTI_FINAL_ICE_MAP.tif"),
        "NEW_LOLA_DEM (LBL)": os.path.join(BASE_DIR, "LDEM_85S_10M.LBL"),
        "NEW_AVG_ILLUM": os.path.join(BASE_DIR, "AVGVISIB_85S_060M_201608.tiff"),
        "NEW_PSR_MASK": os.path.join(BASE_DIR, "LPSR_85S_060M_201608.tiff")
    }
    
    radar_bounds = None
    
    for name, fpath in files.items():
        if not os.path.exists(fpath):
            print(f"\n[{name}] FILE NOT FOUND: Check path -> {fpath}")
            continue
            
        try:
            with rasterio.open(fpath) as src:
                b = src.bounds
                crs = src.crs
                
                print(f"\n{name}")
                print(f"  Bounds: Left={b.left:.1f}, Right={b.right:.1f}, Bottom={b.bottom:.1f}, Top={b.top:.1f}")
                print(f"  Size:   {src.width} x {src.height} pixels")
                print(f"  CRS:    {crs.data['proj'] if crs and 'proj' in crs.data else crs}")
                
                if name == "DFSAR_RADAR_SWATH":
                    radar_bounds = b
                elif radar_bounds:
                    # Check if the new dataset completely encompasses the radar swath
                    covers_x = (b.left <= radar_bounds.left) and (b.right >= radar_bounds.right)
                    covers_y = (b.bottom <= radar_bounds.bottom) and (b.top >= radar_bounds.top)
                    
                    if covers_x and covers_y:
                        print("  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.")
                    else:
                        print("  ❌ STATUS: Does NOT fully encompass the Radar Swath. Data gaps will occur.")
        except Exception as e:
            print(f"\n[{name}] ERROR reading file: {e}")

# Execute Phase 1
spatial_overlap_audit_v2()

PHASE 1: SPATIAL OVERLAP AUDIT (Raw Source vs Radar Swath)

DFSAR_RADAR_SWATH
  Bounds: Left=-27473.8, Right=127476.2, Bottom=12328.7, Top=90053.7
  Size:   6198 x 3109 pixels
  CRS:    stere

ICE_TARGETS
  Bounds: Left=-27473.8, Right=127476.2, Bottom=12328.7, Top=90053.7
  Size:   6198 x 3109 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_LOLA_DEM (LBL)
  Bounds: Left=-151680.0, Right=151680.0, Bottom=-151680.0, Top=151680.0
  Size:   30336 x 30336 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_AVG_ILLUM
  Bounds: Left=-151740.0, Right=151740.0, Bottom=-151740.0, Top=151740.0
  Size:   5058 x 5058 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_PSR_MASK
  Bounds: Left=-151740.0, Right=151740.0, Bottom=-151740.0, Top=151740.0
  Size:   5058 x 5058 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Rad

### Phase 2: Safe Ingestion, Scaling, and Master Grid Warping

### 1. Scientific Rationale & Core Physics
Raw planetary data arrays are distributed in specialized binary formats. For the Lunar Orbiter Laser Altimeter (LOLA), data is stored as a detached ASCII text label (`.LBL`) which maps to an uncompressed 16-bit binary image (`.IMG`). 
Attempting to parse a raw `.IMG` file directly via standard image processors causes an immediate I/O error because it lacks spatial coordinate headers. Furthermore, the values stored inside are Digital Numbers (DN) rather than metrics in meters. According to the PDS3 label metadata:
$$\text{Elevation (meters)} = \text{DN} \times 0.5$$
If this scaling property is not applied directly upon array ingestion, the relative topography flatlines by exactly 50%, resulting in invalid derivative gradients during slope calculation.

##### 2. Algorithmic Strategy

We isolate the coordinate reference system (CRS) and precise affine transform matrix from our master radar swath (`WARPED_CPR.tif`). The raw binary numbers are extracted by opening the `.LBL` file (letting GDAL parse the binary pointer natively), multiplied by the `0.5` physical scaling factor, and mathematically projected onto the master 25-meter grid. Out-of-bounds space is strictly encoded to a floating-point `NODATA = -9999.0`. 

For the solar illumination data (`AVGVISIB`) and the permanent shadow mask (`LPSR`), which have a native spacing of 60 meters per pixel, we warp them into the same grid using `Resampling.bilinear` and `Resampling.nearest` respectively to ensure crisp spatial coregistration without blurring categorical masks.

In [2]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

# Define absolute filepaths based on verified audit locations
LDEM_LBL_PATH = os.path.join(BASE_DIR, "LDEM_85S_10M.LBL")
AVGVISIB_PATH = os.path.join(BASE_DIR, "AVGVISIB_85S_060M_201608.tiff")
LPSR_PATH     = os.path.join(BASE_DIR, "LPSR_85S_060M_201608.tiff")
REF_GRID_PATH = os.path.join(BASE_DIR, "WARPED_CPR.tif")

# Target Master Phase 2 Output GeoTIFFs
OUT_WARPED_DEM   = os.path.join(BASE_DIR, "LDEM85S_WARPED_25m.tif")
OUT_WARPED_ILLUM = os.path.join(BASE_DIR, "AVGVISIB_WARPED_25m.tif")
OUT_WARPED_PSR   = os.path.join(BASE_DIR, "PSR_MASK_25m.tif")

def execute_phase2_ingest():
    print("=" * 70)
    print("PHASE 2: EXECUTING NATIVE PDS/PGDA INGESTION & GRID ALIGNMENT")
    print("=" * 70)
    
    # Extract authoritative spatial framework from Phase 1 radar baseline
    with rasterio.open(REF_GRID_PATH) as ref:
        dst_transform = ref.transform
        dst_crs       = ref.crs
        dst_shape     = (ref.height, ref.width)
        master_profile = ref.profile.copy()
    
    print(f"Target Sync Framework: {dst_shape[1]}x{dst_shape[0]} matrix at 25.0m resolution.")
    
    # ----------------------------------------------------------------
    # LAYER 1: NATIVE PDS3 ELEVATION MODEL (LDEM)
    # ----------------------------------------------------------------
    print("\nProcessing Topographic Ingestion layer...")
    # Open the .LBL file. Rasterio/GDAL uses the label metadata to parse the .IMG binary stream.
    with rasterio.open(LDEM_LBL_PATH) as src:
        # Read the raw binary table into memory as float32 to hold decimal precision after scaling
        raw_dn = src.read(1).astype(np.float32)
        
        # Enforce the explicit scaling multiplier dictated by NASA metadata
        print("Applying planetary scaling modifier: DN * 0.5 meters...")
        scaled_dem = raw_dn * 0.5
        
        # Allocate clean destination canvas pre-padded with un-traversable NoData bounds
        dem_dst_array = np.full(dst_shape, NODATA, dtype=np.float32)
        
        print("Executing bilinear surface grid projection...")
        reproject(
            source=scaled_dem,
            destination=dem_dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )
        
    # Write the clean, uncorrupted elevation master raster
    master_profile.update(dtype=rasterio.float32, count=1, compress='lzw', nodata=NODATA)
    with rasterio.open(OUT_WARPED_DEM, 'w', **master_profile) as dst:
        dst.write(dem_dst_array, 1)
    print(f"[SUCCESS] Elevation Layer written: {OUT_WARPED_DEM}")
    
    # ----------------------------------------------------------------
    # LAYER 2: TIME-INTEGRATED ANNUAL ILLUMINATION (AVGVISIB)
    # ----------------------------------------------------------------
    print("\nProcessing Solar Illumination layer...")
    with rasterio.open(AVGVISIB_PATH) as src:
        raw_illum = src.read(1).astype(np.float32)
        
        # Clean any default system nodata boundaries from PGDA
        if src.nodata is not None:
            raw_illum[raw_illum == src.nodata] = np.nan
            
        illum_dst_array = np.full(dst_shape, NODATA, dtype=np.float32)
        
        print("Executing continuous bilinear downsampling from 60m to 25m...")
        reproject(
            source=raw_illum,
            destination=illum_dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )
        
    with rasterio.open(OUT_WARPED_ILLUM, 'w', **master_profile) as dst:
        dst.write(illum_dst_array, 1)
    print(f"[SUCCESS] Solar Illumination written: {OUT_WARPED_ILLUM}")

    # ----------------------------------------------------------------
    # LAYER 3: BINARY CUMULATIVE PSR MASK (LPSR)
    # ----------------------------------------------------------------
    print("\nProcessing Permanently Shadowed Region mask...")
    with rasterio.open(LPSR_PATH) as src:
        raw_psr = src.read(1).astype(np.float32)
        
        psr_dst_array = np.full(dst_shape, NODATA, dtype=np.float32)
        
        # CRITICAL MARGIN: We use nearest-neighbor for a binary mask to protect
        # the sharp 0/1 boolean structure. Bilinear would introduce fractional data blur.
        print("Executing categorical nearest-neighbor allocation...")
        reproject(
            source=raw_psr,
            destination=psr_dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )
        
    with rasterio.open(OUT_WARPED_PSR, 'w', **master_profile) as dst:
        dst.write(psr_dst_array, 1)
    print(f"[SUCCESS] PSR Binary Mask written: {OUT_WARPED_PSR}")

# Execute Phase 2 Ingestion Block
execute_phase2_ingest()

PHASE 2: EXECUTING NATIVE PDS/PGDA INGESTION & GRID ALIGNMENT
Target Sync Framework: 6198x3109 matrix at 25.0m resolution.

Processing Topographic Ingestion layer...
Applying planetary scaling modifier: DN * 0.5 meters...
Executing bilinear surface grid projection...
[SUCCESS] Elevation Layer written: C:\DRISHTI_POC\LDEM85S_WARPED_25m.tif

Processing Solar Illumination layer...
Executing continuous bilinear downsampling from 60m to 25m...
[SUCCESS] Solar Illumination written: C:\DRISHTI_POC\AVGVISIB_WARPED_25m.tif

Processing Permanently Shadowed Region mask...
Executing categorical nearest-neighbor allocation...
[SUCCESS] PSR Binary Mask written: C:\DRISHTI_POC\PSR_MASK_25m.tif
